# Step 4: All 90 features for the LightGBM ranker

This notebook produces **all 90 features** of the proposal (families A to H plus user context; see `JDsearch_Part3_Feature_Reference.xlsx`). It saves them in LightGBM-ready Parquet files, one per split. It also provides a **feature loader**, in which you choose the features to load for training by typing their names. It does not train a model. The final text cell is a guide to training LightGBM on the loaded data.

All features are computed with step_3_person_match_feature_engineering.ipynb's code and settings, so they are identical to step_3_person_match_feature_engineering.ipynb's. The feature computation is step_3_person_match_feature_engineering.ipynb's `build_chunk_features`, copied unchanged.

**Reusing step_3_person_match_feature_engineering.ipynb's saved work.** The slow parts of step_3_person_match_feature_engineering.ipynb were building the title TF-IDF matrix and training the item2vec and term2vec embeddings. step_3_person_match_feature_engineering.ipynb saved all of them in `processed/part3_features/intermediates_backup/`, and this notebook reuses them, together with the session chunks and priors when they are present:

| Artifact | Files in `intermediates_backup/` | Used for | If missing or inconsistent |
|---|---|---|---|
| Session chunks and split | `stage/cand_*.parquet`, `stage/hist_*.parquet`, `sessions.parquet`, `step1_stats.json` | every feature | rebuilt from `user_behavior_data.txt` (step_3_person_match_feature_engineering.ipynb Step 1 code) |
| Product tables | `arrays/lookup_wid.npy`, `lookup_row.npy`, `prod_attr.npy`, `prod_meta.json`, `title_vocab.parquet`, `title_indptr.npy`, `title_indices.bin`, `title_data.bin` | attribute, title and query-overlap features | rebuilt from `product_meta_data.txt` (step_3_person_match_feature_engineering.ipynb Step 3 code) |
| Item2vec vectors | `arrays/i2v_keys.npy`, `arrays/i2v_vecs.npy` | `i2v_*`, `last_i2v_cos` | retrained with step_3_person_match_feature_engineering.ipynb's settings |
| Term2vec vectors | `arrays/t2v_term_vecs.npy` | `t2v_cos_profile` | retrained from the title cache (`titles/`), which is itself rebuilt from the product file if absent |
| Training-set priors | `stage/priors_*.parquet` | family G and the `item_ctr_train_slate_*` features | recomputed (step_3_person_match_feature_engineering.ipynb Step 5 code) |

Each reused artifact is **validated before use**: its files must exist and their sizes must agree with one another. Anything missing or inconsistent is rebuilt automatically and, if `SAVE_REBUILT_TO_DRIVE` is `True`, saved back to Drive for later runs.

**Outputs.** These go in `processed/lgbm_features_all/` on Drive, laid out as storage option A:

| File | Contents |
|---|---|
| `train.parquet`, `val.parquet`, `test.parquet` | One row per candidate: `session_id` (int32), `wid` (int64), `label` (int8, 0–3) and the 90 features (float32; `NaN` = no evidence). Rows are sorted by session, with each session's candidates contiguous. zstd compression. |
| `train_groups.npy`, `val_groups.npy`, `test_groups.npy` | Number of candidates per session, in row order: LightGBM's `group` argument |
| `feature_list.json` | The 90 feature names, in column order |
| `feature_dictionary_part3.csv` | Family, definition and meaning of a missing value, for each feature |
| `MERGE_COMPLETE.json` | Written last; its presence means the three split files are complete |

The split files take roughly 2.5–4 GB on Drive. While they are being written, each chunk file is deleted as soon as it has been appended, so the data never exist twice in full.

**Memory.** The large tables are read memory-mapped from local disk, and features are computed 2,000 sessions at a time and written out immediately. The loader fills a preallocated array instead of concatenating copies. A run interrupted by a runtime limit resumes at the first missing chunk.

**Running order.** For a fresh build, run Steps 0 to 8 in order. To train later from the saved files, only Step 0's first cell (mounting Drive) and Step 9 (the loader) are needed.

**Note on printed output.** Some printed lines and the "Part 3 AUC" column in Step 8 still say "Part 3": this refers to `step_3_person_match_feature_engineering.ipynb`.

In [6]:
%pip install -q gensim==4.4.0

## Step 0 — Setup

All settings below must stay identical to step_3_person_match_feature_engineering.ipynb's, because the reused files were built with them. `SAVE_REBUILT_TO_DRIVE` controls whether anything rebuilt here is copied back to the step_3_person_match_feature_engineering.ipynb backup folder. `DELETE_EACH_CHUNK_AFTER_MERGE` keeps the Drive footprint low in Step 7.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, math, os, shutil, time
from collections import Counter
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import psutil
    def ram(label=""):
        print(f"[RAM] {label}: {psutil.Process().memory_info().rss / 1e9:.2f} GB in use")
except ImportError:
    def ram(label=""):
        pass

# ---------------- paths ----------------
PROJECT_DIR = Path("/content/drive/MyDrive/BT4222 Final/JDSearch")
RAW_DIR = PROJECT_DIR / "Raw Data"
USER_TXT = RAW_DIR / "user_behavior_data.txt"
PRODUCT_TXT = RAW_DIR / "product_meta_data.txt"
PRODUCT_CACHE_DIR = RAW_DIR / "product_chunks"
OUTPUT_DIR = PROJECT_DIR / "processed"
BACKUP_DIR = OUTPUT_DIR / "part3_features" / "intermediates_backup"   # step_3_person_match_feature_engineering.ipynb's saved intermediates
OUT_DIR = OUTPUT_DIR / "lgbm_features_all"                           # final outputs of this notebook
CHUNK_DIR = OUT_DIR / "chunks"                                       # per-chunk features (temporary)
MERGE_DONE = OUT_DIR / "MERGE_COMPLETE.json"
P3_DIR = OUT_DIR                                                     # name used by step_3_person_match_feature_engineering.ipynb's dictionary code
LOCAL_DIR = Path("/content/part5_local")                             # local working copies
STAGE_DIR, ARR_DIR, TITLE_DIR = LOCAL_DIR / "stage", LOCAL_DIR / "arrays", LOCAL_DIR / "titles"
for d in [OUT_DIR, CHUNK_DIR, STAGE_DIR, ARR_DIR, TITLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAVE_REBUILT_TO_DRIVE = True          # copy anything rebuilt here back to BACKUP_DIR
DELETE_EACH_CHUNK_AFTER_MERGE = True  # delete each chunk file as soon as it has been appended in Step 7

# ---------------- settings: identical to step_3_person_match_feature_engineering.ipynb ----------------
SEED = 42
SESSIONS_PER_CHUNK = 2000
READ_ROWS = 10_000
CART_MAX_DAYS = 90
TYPE_CODE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
TYPE_NAMES = ["click", "cart", "ord", "flw"]
TYPE_WEIGHT = np.array([1.0, 3.0, 5.0, 2.0])
TAU_DAYS = 30.0
TERM_SEP = "\x18"
TITLE_MIN_DF = 5
EMB_DIM, EMB_WINDOW, EMB_EPOCHS = 64, 5, 5
I2V_MIN_COUNT = 5
PRIOR_ALPHA = 20.0
N_FOLDS = 5
BURST_GAP_DAYS = 30 / 1440
SCREEN_SESSIONS = 10_000
EXPECTED = {"sessions": 173_831, "eligible": 171_175, "candidate_rows": 15_348_049, "carts_removed": 1_177}


# ---------------- backup helpers ----------------
def restore(*names):
    # Copy files/folders from the Drive backup to local disk where missing; True if all exist locally.
    ok = True
    for n in names:
        dst, src = LOCAL_DIR / n, BACKUP_DIR / n
        if src.is_dir():
            for f in src.rglob("*"):
                t = dst / f.relative_to(src)
                if f.is_file() and not t.exists():
                    t.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, t)
        elif src.exists() and not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        ok = ok and dst.exists() and (not dst.is_dir() or any(dst.iterdir()))
    return ok


def backup(*names):
    # Copy rebuilt files/folders from local disk to the Drive backup (if SAVE_REBUILT_TO_DRIVE).
    if not SAVE_REBUILT_TO_DRIVE:
        return
    for n in names:
        src, dst = LOCAL_DIR / n, BACKUP_DIR / n
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)


def mark_done(flag):
    (LOCAL_DIR / f"{flag}.done").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


ram("setup")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[RAM] setup: 0.25 GB in use


### The feature list

The 90 features, grouped by family, are defined once here (step_3_person_match_feature_engineering.ipynb code, unchanged). The feature computation, the saved column order and the loader all refer to this list.

In [8]:
ATTRS = ["brand", "shop", "cate1", "cate2", "cate3", "cate4"]       # order of the attribute columns
ATTR_SOURCE = ["brand_id", "shop_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4"]
BRAND, CATE3 = 0, 4                                                  # column positions used by E, F, G, H
H_BASE = ["brand_aff", "shop_aff", "cate3_aff", "title_cos_profile", "title_cos_max", "i2v_cos_max", "item_ctr_train"]

FAMILIES = {
    "A": ["exact_wid_click_0_7d", "exact_wid_cart_0_7d", "exact_wid_ord_0_7d", "exact_wid_cart_7_90d",
          "exact_wid_ord_7_90d", "exact_wid_ord_90_365d", "exact_wid_last_seen_days"],
    "B": [f"{k}_{s}" for k in ATTRS for s in ["aff", "share", "last_seen_days", "ord_cnt"]],
    "C": ["title_cos_profile", "title_cos_profile_ord", "title_cos_max", "title_hist_qry_overlap"],
    "D": ["i2v_cos_mean", "i2v_cos_max", "i2v_cos_last5", "t2v_cos_profile"],
    "E": ["last_same_brand", "last_same_cate3", "last_title_cos", "last_i2v_cos"],
    "F": ["qhist_n_actions", "qhist_wid_cnt", "qhist_brand_aff", "qhist_cate3_aff"],
    "G": ["item_ctr_train", "brand_ctr_train", "cate3_ctr_train", "item_impr_train"],
    "H": [f"{f}_slate_{s}" for f in H_BASE for s in ["pct", "z"]] + ["cate3_slate_share", "slate_size"],
    "context": [f"{t}_count_{w}" for w in ["0_7d", "7_90d", "90_365d"] for t in TYPE_NAMES]
               + ["history_len_clean", "history_span_days", "days_since_last_interaction",
                  "share_click", "share_cart", "share_ord", "share_flw",
                  "cate3_entropy", "n_distinct_brands", "qryless_share", "burst_len"],
}
FEATURE_COLUMNS = [f for fam in FAMILIES.values() for f in fam]
FAMILY_OF = {f: fam for fam, cols in FAMILIES.items() for f in cols}
ID_COLUMNS = ["session_id", "wid", "label", "split"]
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 90
print({k: len(v) for k, v in FAMILIES.items()}, "total", len(FEATURE_COLUMNS))

{'A': 7, 'B': 24, 'C': 4, 'D': 4, 'E': 4, 'F': 4, 'G': 4, 'H': 16, 'context': 23} total 90


### Inventory of the step_3_person_match_feature_engineering.ipynb backup

This cell lists what is available in `intermediates_backup/`, so you can see before anything runs which steps will be reused and which rebuilt. It only reports; the following steps make the decisions.

In [9]:
PRODUCT_FILES = ["lookup_wid.npy", "lookup_row.npy", "prod_attr.npy", "prod_meta.json", "title_vocab.parquet",
                 "title_indptr.npy", "title_indices.bin", "title_data.bin"]
I2V_FILES = ["i2v_keys.npy", "i2v_vecs.npy"]
T2V_FILES = ["t2v_term_vecs.npy"]


def size_gb(p):
    if not p.exists():
        return None
    return (sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size) / 1e9


rows = []
for group, names in [("session chunks", ["stage", "sessions.parquet", "step1_stats.json"]),
                     ("product tables", [f"arrays/{n}" for n in PRODUCT_FILES]),
                     ("item2vec", [f"arrays/{n}" for n in I2V_FILES]),
                     ("term2vec", [f"arrays/{n}" for n in T2V_FILES]),
                     ("title cache (only needed to retrain term2vec)", ["titles"])]:
    for n in names:
        s = size_gb(BACKUP_DIR / n)
        rows.append({"artifact": group, "file": n, "in backup": s is not None, "size (GB)": None if s is None else round(s, 3)})
display(pd.DataFrame(rows))
stage_bk = BACKUP_DIR / "stage"
print("Backed-up chunk files:", {k: len(list(stage_bk.glob(f"{k}_*.parquet"))) if stage_bk.exists() else 0
                                 for k in ["cand", "hist", "priors"]})
print("Raw files present: user file", USER_TXT.exists(), "| product file", PRODUCT_TXT.exists() or PRODUCT_CACHE_DIR.exists())
print("Final outputs already complete:", MERGE_DONE.exists())

,artifact,file,in backup,size (GB)
0,session chunks,stage,True,0.696
1,session chunks,sessions.parquet,True,0.004
2,session chunks,step1_stats.json,True,0.000
3,product tables,arrays/lookup_wid.npy,True,0.092
4,product tables,arrays/lookup_row.npy,True,0.092
5,product tables,arrays/prod_attr.npy,True,0.552
6,product tables,arrays/prod_meta.json,True,0.000
7,product tables,arrays/title_vocab.parquet,True,0.002
8,product tables,arrays/title_indptr.npy,True,0.092
9,product tables,arrays/title_indices.bin,True,1.073


Backed-up chunk files: {'cand': 86, 'hist': 86, 'priors': 86}
Raw files present: user file True | product file True
Final outputs already complete: False


## Step 1 — Session chunks and the split

**What this step does.** It restores step_3_person_match_feature_engineering.ipynb's session chunks from the backup: candidates (`cand_*`), cleaned histories (`hist_*`), the session table and the Step 1 statistics. It checks that a candidate and a history file exist for every chunk, and that the number of candidate rows matches the statistics. If not, the chunks are rebuilt from `user_behavior_data.txt` with step_3_person_match_feature_engineering.ipynb's Step 1 code, reproduced unchanged below. That code:
- parses the candidates and labels and keeps the eligible sessions;
- shuffles each session's candidates with seed 42;
- parses and cleans the histories (elapsed days; carts older than 90 days removed);
- writes chunks of 2,000 sessions.

The seed is fixed, so a rebuild produces exactly the same chunks. The split is then reproduced with the step_1_data_preprocessing.ipynb method (reproduced unchanged in step_3_person_match_feature_engineering.ipynb Step 2).

In [ ]:
def split_flat(series, sep):
    # Split each string of a Series on `sep`; return the flat token array and the number of tokens per row.
    vals = series.fillna("").to_numpy(dtype=object)
    lists = [v.split(sep) if v else [] for v in vals]
    lens = np.fromiter((len(l) for l in lists), dtype=np.int64, count=len(lists))
    flat = np.array(list(chain.from_iterable(lists)), dtype=object)
    return flat, lens


def within_pos(lens):
    # Position of each token within its own row (0, 1, 2, ... restarting for every row).
    starts = np.cumsum(lens) - lens
    return np.arange(lens.sum(), dtype=np.int64) - np.repeat(starts, lens)


def to_int64(tokens, what):
    # Convert ID tokens to int64; stop with a clear message if any ID is not an integer.
    s = pd.to_numeric(pd.Series(tokens, dtype=object), errors="coerce")
    if s.isna().any():
        bad = pd.Series(tokens)[s.isna()].head(3).tolist()
        raise ValueError(f"Non-integer {what} found, e.g. {bad}. The notebook stores IDs as integers to save memory.")
    return s.to_numpy(dtype=np.int64)


def select_first(flat, lens, keep_n):
    # Keep the first keep_n[i] tokens of row i (the truncation that step_1_data_preprocessing.ipynb's zip performs).
    pos = within_pos(lens)
    return flat[pos < np.repeat(keep_n, lens)]


def parse_block(df, sid0):
    # Parse one block of raw rows. Returns candidate rows, cleaned history rows and session rows
    # for the eligible sessions only, plus counters for the printed checks.
    n = len(df)
    local = np.arange(n)
    stats = Counter()

    # ---- candidates and eligibility (step_1_data_preprocessing.ipynb) ----
    cw, cn = split_flat(df["candidate_wid_list"], "_")
    cl, cln = split_flat(df["candidate_label_list"], "_")
    lab = pd.to_numeric(pd.Series(cl, dtype=object)).to_numpy()
    l_local = np.repeat(local, cln)
    n_pos = np.bincount(l_local, weights=(lab > 0), minlength=n)
    n_neg = np.bincount(l_local, weights=(lab == 0), minlength=n)
    eligible = (n_pos > 0) & (n_neg > 0)
    # as in step_1_data_preprocessing.ipynb, IDs and labels must be aligned in every eligible session
    assert (cn == cln)[eligible].all(), "candidate IDs and labels are misaligned in an eligible session"
    c_local = np.repeat(local, cn)
    lab = lab[eligible[l_local]]
    stats["sessions"] += n
    stats["eligible"] += int(eligible.sum())
    stats["all_negative"] += int((n_pos == 0).sum())
    stats["all_positive"] += int((n_neg == 0).sum())

    keep_c = eligible[c_local]
    cand = pd.DataFrame({
        "session_id": (sid0 + c_local[keep_c]).astype(np.int32),
        "wid": to_int64(cw[keep_c], "candidate product ID"),
        "label": lab.astype(np.int8),
        "position": within_pos(cn)[keep_c].astype(np.int16),
    })

    # ---- history: parse, align, elapsed days (step_1_data_preprocessing.ipynb convention) ----
    hw, hwn = split_flat(df["history_wid_list"], "_")
    ht, htn = split_flat(df["history_type_list"], "_")
    hq, hqn = split_flat(df["history_qry_list"], "_")
    tm, tmn = split_flat(df["history_time_list"], "_")
    t_all = to_int64(tm, "time gap")
    t_pos = within_pos(tmn)
    gaps = t_all[t_pos >= 1]                          # entries 1..n of each row
    g_len = np.maximum(tmn - 1, 0)
    rev = np.cumsum(gaps[::-1])[::-1]                 # sum of this and all later gaps (across rows)
    after = np.append(rev, 0)[np.cumsum(g_len)]       # part of that sum belonging to later rows
    days_all = (rev - np.repeat(after, g_len)) / 86400.0
    has_t = tmn > 0
    last_gap = t_all[(np.cumsum(tmn) - 1)[has_t]]
    stats["rows_with_times"] += int(eligible[has_t].sum())
    stats["negative_last_gap"] += int((last_gap < 0)[eligible[has_t]].sum())

    n_act = np.minimum(np.minimum(hwn, htn), g_len)   # step_1_data_preprocessing.ipynb zips wid, type and days
    h_wid = to_int64(select_first(hw, hwn, n_act), "history product ID")
    h_type = pd.Series(select_first(ht, htn, n_act)).map(TYPE_CODE).fillna(-1).to_numpy(np.int8)
    h_days = select_first(days_all, g_len, n_act)
    # past queries: pad with '-1' where the query list is shorter
    out_start = np.cumsum(n_act) - n_act
    h_qry = np.full(int(n_act.sum()), "-1", dtype=object)
    q_pos = within_pos(hqn)
    q_ok = q_pos < np.repeat(n_act, hqn)
    h_qry[np.repeat(out_start, hqn)[q_ok] + q_pos[q_ok]] = hq[q_ok]
    h_local = np.repeat(local, n_act)

    # cleaning (step_1_data_preprocessing.ipynb): remove carts older than 90 days; then keep eligible sessions
    old_cart = (h_type == TYPE_CODE["CART"]) & (h_days > CART_MAX_DAYS)
    keep_h = eligible[h_local] & ~old_cart
    stats["carts_removed"] += int((old_cart & eligible[h_local]).sum())
    is_click = h_type[keep_h] == TYPE_CODE["CLICK"]
    stats["clicks"] += int(is_click.sum())
    stats["clicks_within_7d"] += int((h_days[keep_h][is_click] < 7).sum())
    hist = pd.DataFrame({
        "session_id": (sid0 + h_local[keep_h]).astype(np.int32),
        "wid": h_wid[keep_h],
        "type": h_type[keep_h],
        "days": h_days[keep_h].astype(np.float32),
        "hqry": h_qry[keep_h],
    })
    sess = pd.DataFrame({
        "session_id": (sid0 + local[eligible]).astype(np.int32),
        "query": df["query"].fillna("").to_numpy(dtype=object)[eligible],
        "slate_size": cn[eligible].astype(np.int32),
    })
    return cand, hist, sess, stats

In [ ]:
def stage_ok():
    # Restore step_3_person_match_feature_engineering.ipynb's session chunks and check that they are complete and consistent.
    if not restore("stage", "sessions.parquet", "step1_stats.json"):
        print("Session chunks not in the backup - rebuilding from the raw user file.")
        return False
    st = json.loads((LOCAL_DIR / "step1_stats.json").read_text())
    n = int(st["chunks"])
    files_ok = all((STAGE_DIR / f"{k}_{i:03d}.parquet").exists() for i in range(n) for k in ["cand", "hist"])
    rows = sum(pq.ParquetFile(STAGE_DIR / f"cand_{i:03d}.parquet").metadata.num_rows for i in range(n)) if files_ok else -1
    if files_ok and rows == st["candidate_rows"]:
        print(f"Reusing step_3_person_match_feature_engineering.ipynb's session chunks: {n} chunks, {rows:,} candidate rows.")
        return True
    print("Backed-up session chunks are incomplete - rebuilding from the raw user file.")
    shutil.rmtree(STAGE_DIR, ignore_errors=True); STAGE_DIR.mkdir(parents=True)
    return False


def write_chunk(chunk_id, cand, hist, rng):
    # Shuffle candidates within each session (display order is not kept as row order) and write both files.
    order = np.lexsort((rng.random(len(cand)), cand["session_id"].to_numpy()))
    cand = cand.iloc[order].reset_index(drop=True)
    cand.to_parquet(STAGE_DIR / f"cand_{chunk_id:03d}.parquet", index=False)
    hist.to_parquet(STAGE_DIR / f"hist_{chunk_id:03d}.parquet", index=False)
    return np.unique(cand["wid"].to_numpy()), np.unique(hist["wid"].to_numpy())


if stage_ok():
    print("Step 1 already completed - skipping.")
else:
    rng = np.random.default_rng(SEED)
    totals = Counter()
    pend_c, pend_h, pend_s = [], [], []
    sess_rows, needed = [], []
    chunk_id, sid0 = 0, 0
    reader = pd.read_csv(USER_TXT, sep="\t", dtype=str, chunksize=READ_ROWS)
    for block in reader:
        cand, hist, sess, st = parse_block(block, sid0)
        sid0 += len(block)
        totals.update(st)
        pend_c.append(cand); pend_h.append(hist); pend_s.append(sess)
        n_pending = sum(len(s) for s in pend_s)
        while n_pending >= SESSIONS_PER_CHUNK:
            s_all = pd.concat(pend_s, ignore_index=True)
            c_all = pd.concat(pend_c, ignore_index=True)
            h_all = pd.concat(pend_h, ignore_index=True)
            cut = s_all["session_id"].iloc[SESSIONS_PER_CHUNK - 1]
            s_out = s_all[s_all["session_id"] <= cut].assign(chunk=chunk_id)
            uc, uh = write_chunk(chunk_id, c_all[c_all["session_id"] <= cut], h_all[h_all["session_id"] <= cut], rng)
            sess_rows.append(s_out); needed += [uc, uh]
            pend_s = [s_all[s_all["session_id"] > cut]]
            pend_c = [c_all[c_all["session_id"] > cut]]
            pend_h = [h_all[h_all["session_id"] > cut]]
            chunk_id += 1
            n_pending = len(pend_s[0])
        print(f"read {sid0:,} rows -> {chunk_id} chunks written", end="\r")
        del block, cand, hist, sess
    # final partial chunk
    s_all = pd.concat(pend_s, ignore_index=True)
    if len(s_all):
        uc, uh = write_chunk(chunk_id, pd.concat(pend_c, ignore_index=True), pd.concat(pend_h, ignore_index=True), rng)
        sess_rows.append(s_all.assign(chunk=chunk_id)); needed += [uc, uh]
        chunk_id += 1
    sessions = pd.concat(sess_rows, ignore_index=True)
    sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
    np.save(ARR_DIR / "needed_wids.npy", np.unique(np.concatenate(needed)))
    totals["chunks"] = chunk_id
    totals["candidate_rows"] = int(sessions["slate_size"].sum())
    (LOCAL_DIR / "step1_stats.json").write_text(json.dumps({k: int(v) for k, v in totals.items()}))
    del pend_c, pend_h, pend_s, sess_rows, needed, s_all, sessions
    gc.collect()
    mark_done("step1")
    backup("stage", "sessions.parquet", "arrays/needed_wids.npy", "step1_stats.json")

stats1 = json.loads((LOCAL_DIR / "step1_stats.json").read_text())
sessions = pd.read_parquet(LOCAL_DIR / "sessions.parquet")
N_CHUNKS = int(stats1["chunks"])
print(f"Sessions in file:            {stats1['sessions']:,} (expected {EXPECTED['sessions']:,})")
print(f"Eligible sessions:           {stats1['eligible']:,} (expected {EXPECTED['eligible']:,})")
print(f"  excluded all-negative:     {stats1['all_negative']:,};  all-positive: {stats1['all_positive']:,}")
print(f"Candidate rows:              {stats1['candidate_rows']:,} (expected {EXPECTED['candidate_rows']:,})")
print(f"Chunks of {SESSIONS_PER_CHUNK} sessions:      {N_CHUNKS}")
print(f"Cart actions removed (>90d): {stats1['carts_removed']:,} (expected {EXPECTED['carts_removed']:,})")
print(f"Check - negative final gaps: {stats1['negative_last_gap'] / max(stats1['rows_with_times'], 1):.4%} of sessions (should be 0)")
print(f"Check - clicks within 7 days: {stats1['clicks_within_7d'] / max(stats1['clicks'], 1):.2%} (should be close to 100%)")
ram("after step 1")

Reusing Part 3's session chunks: 86 chunks, 15,348,049 candidate rows.
Step 1 already completed - skipping.
Sessions in file:            173,831 (expected 173,831)
Eligible sessions:           171,175 (expected 171,175)
  excluded all-negative:     2,103;  all-positive: 553
Candidate rows:              15,348,049 (expected 15,348,049)
Chunks of 2000 sessions:      86
Cart actions removed (>90d): 1,177 (expected 1,177)
Check - negative final gaps: 0.0000% of sessions (should be 0)
Check - clicks within 7 days: 100.00% (should be close to 100%)
[RAM] after step 1: 0.32 GB in use


In [12]:
from sklearn.model_selection import train_test_split

ids = pd.Series(sessions["session_id"].to_numpy(), name="session_id")
train_ids, temp_ids = train_test_split(ids, test_size=0.20, random_state=SEED, shuffle=True)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED, shuffle=True)
split_map = pd.Series("train", index=ids.to_numpy())
split_map.loc[val_ids.to_numpy()] = "val"
split_map.loc[test_ids.to_numpy()] = "test"
sessions["split"] = split_map.reindex(sessions["session_id"].to_numpy()).to_numpy()
sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
backup("sessions.parquet")
print(sessions["split"].value_counts())
del ids, train_ids, temp_ids, val_ids, test_ids

split
train    136940
test      17118
val       17117
Name: count, dtype: int64


## Step 2 — Product tables (attributes and title TF-IDF)

**What this step does.** It restores step_3_person_match_feature_engineering.ipynb's product tables and checks that their sizes agree:
- the lookup arrays have equal length;
- the attribute table has one row per product plus the empty row;
- the TF-IDF row pointers have one entry per product plus two;
- the index and data files each hold exactly the number of entries the row pointers announce;
- the vocabulary has the size recorded in `prod_meta.json`.

If any check fails, the tables are rebuilt from the product metadata with step_3_person_match_feature_engineering.ipynb's Step 3 code, reproduced unchanged below. That code keeps the products occurring as candidates or in histories, stores brand, shop and four category levels as integers, and builds unit-length TF-IDF title vectors from terms in at least 5 titles. A rebuild also recreates the title text cache, which Step 3 needs only if term2vec has to be retrained.

The helper cell that follows opens the tables memory-mapped (step_3_person_match_feature_engineering.ipynb code, unchanged).

In [ ]:
def product_tables_ok():
    # Restore step_3_person_match_feature_engineering.ipynb's product tables and check that their sizes agree with one another.
    if not restore(*[f"arrays/{n}" for n in PRODUCT_FILES]):
        print("Product tables not (fully) in the backup - rebuilding.")
        return False
    try:
        meta = json.loads((ARR_DIR / "prod_meta.json").read_text())
        indptr = np.load(ARR_DIR / "title_indptr.npy", mmap_mode="r"); nnz = int(indptr[-1])
        checks = {
            "lookup arrays equal length": len(np.load(ARR_DIR / "lookup_wid.npy", mmap_mode="r")) == len(np.load(ARR_DIR / "lookup_row.npy", mmap_mode="r")),
            "attribute table rows": np.load(ARR_DIR / "prod_attr.npy", mmap_mode="r").shape == (meta["n_prod"] + 1, 6),
            "TF-IDF row pointers": len(indptr) == meta["n_prod"] + 2,
            "TF-IDF index file complete": (ARR_DIR / "title_indices.bin").stat().st_size == 4 * nnz,
            "TF-IDF data file complete": (ARR_DIR / "title_data.bin").stat().st_size == 4 * nnz,
            "vocabulary size": len(pd.read_parquet(ARR_DIR / "title_vocab.parquet")) == meta["vocab"],
        }
    except Exception as e:
        checks = {f"readable ({type(e).__name__})": False}
    for k, v in checks.items():
        print(f"  {k:30s} {'OK' if v else 'FAILED'}")
    if all(checks.values()):
        print("Reusing step_3_person_match_feature_engineering.ipynb's product tables.")
        return True
    print("Product tables inconsistent - rebuilding.")
    for n in PRODUCT_FILES:
        (ARR_DIR / n).unlink(missing_ok=True)
    return False


def ensure_needed_wids():
    # The product IDs occurring as candidates or in histories (written by Step 1; recomputed if absent).
    f = ARR_DIR / "needed_wids.npy"
    if not f.exists() and not restore("arrays/needed_wids.npy"):
        parts = []
        for i in range(N_CHUNKS):
            for k in ["cand", "hist"]:
                parts.append(np.unique(pd.read_parquet(STAGE_DIR / f"{k}_{i:03d}.parquet", columns=["wid"])["wid"].to_numpy()))
        np.save(f, np.unique(np.concatenate(parts)))
    return np.load(f)


def product_chunks(columns):
    # Yield the product metadata in chunks, from the Parquet cache if present, else from the text file.
    cached = sorted(PRODUCT_CACHE_DIR.glob("*.parquet")) if PRODUCT_CACHE_DIR.exists() else []
    if cached:
        for f in cached:
            yield pd.read_parquet(f, columns=columns)
    else:
        for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", dtype=str, usecols=columns, chunksize=500_000):
            yield chunk


def isin_sorted(values, sorted_ref):
    i = np.searchsorted(sorted_ref, values)
    i = np.minimum(i, len(sorted_ref) - 1)
    return sorted_ref[i] == values


if product_tables_ok():
    print("Step 3 already completed - skipping.")
else:
    needed = ensure_needed_wids()
    wid_parts, attr_parts, n_rows, n_bad = [], [], 0, 0
    for i, ch in enumerate(product_chunks(["wid", "name"] + ATTR_SOURCE)):
        w = pd.to_numeric(ch["wid"], errors="coerce")
        n_bad += int(w.isna().sum())
        keep = w.notna().to_numpy()
        keep[keep] = isin_sorted(w[keep].to_numpy(np.int64), needed)
        ch = ch[keep]
        wid_parts.append(w[keep].to_numpy(np.int64))
        a = np.column_stack([pd.to_numeric(ch[c], errors="coerce").fillna(-1).to_numpy(np.int64) for c in ATTR_SOURCE])
        attr_parts.append(a)
        ch[["name"]].fillna("").to_parquet(TITLE_DIR / f"titles_{i:03d}.parquet", index=False)
        n_rows += len(ch)
        print(f"product chunk {i}: kept {len(ch):,} rows (total {n_rows:,})", end="\r")
    prod_wid = np.concatenate(wid_parts); prod_attr = np.vstack(attr_parts + [np.full((1, 6), -1, np.int64)])
    del wid_parts, attr_parts
    N_PROD = len(prod_wid)
    # lookup: sorted unique IDs -> first row
    order = np.argsort(prod_wid, kind="stable")
    sw = prod_wid[order]
    first = np.ones(len(sw), bool); first[1:] = sw[1:] != sw[:-1]
    np.save(ARR_DIR / "lookup_wid.npy", sw[first]); np.save(ARR_DIR / "lookup_row.npy", order[first].astype(np.int64))
    np.save(ARR_DIR / "prod_attr.npy", prod_attr)
    meta = {"n_prod": int(N_PROD), "duplicates": int((~first).sum()), "non_integer_wid": n_bad,
            "needed": int(len(needed)), "needed_missing": int(len(needed) - first.sum())}
    del prod_wid, prod_attr, order, sw, first, needed
    gc.collect()

    # ---- title TF-IDF: pass 1 counts document frequencies, pass 2 writes the matrix ----
    title_files = sorted(TITLE_DIR.glob("titles_*.parquet"))
    df_count = Counter()
    for f in title_files:
        names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
        df_count.update(chain.from_iterable(set(x.split(TERM_SEP)) - {""} for x in names))
    vocab = sorted(t for t, c in df_count.items() if c >= TITLE_MIN_DF)
    term_df = np.array([df_count[t] for t in vocab], dtype=np.float64)
    idf = np.log((1 + N_PROD) / (1 + term_df)) + 1.0
    vocab_index = pd.Index(vocab)
    pd.DataFrame({"term": vocab}).to_parquet(ARR_DIR / "title_vocab.parquet", index=False)
    del df_count, term_df

    indptr, offset = [np.zeros(1, np.int64)], 0
    with open(ARR_DIR / "title_indices.bin", "wb") as fi, open(ARR_DIR / "title_data.bin", "wb") as fd:
        for f in title_files:
            names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
            toks, lens = split_flat(pd.Series(names), TERM_SEP)
            rows = np.repeat(np.arange(len(names)), lens)
            cols = vocab_index.get_indexer(toks)
            ok = cols >= 0
            m = sp.csr_matrix((np.ones(ok.sum(), np.float32), (rows[ok], cols[ok])), shape=(len(names), len(vocab)))
            m.sum_duplicates()                               # term frequencies (duplicates summed)
            m.data *= idf[m.indices].astype(np.float32)      # tf x idf
            m = normalize(m, norm="l2", copy=False)          # unit length: dot product = cosine
            m.indices.astype(np.int32).tofile(fi)
            m.data.astype(np.float32).tofile(fd)
            indptr.append(offset + m.indptr[1:].astype(np.int64))
            offset += int(m.nnz)
            del names, toks, lens, rows, cols, m
    indptr = np.concatenate(indptr)
    indptr = np.append(indptr, indptr[-1])                   # the extra empty row for unknown products
    np.save(ARR_DIR / "title_indptr.npy", indptr)
    meta.update({"vocab": len(vocab), "title_nnz": int(indptr[-1])})
    (ARR_DIR / "prod_meta.json").write_text(json.dumps(meta))
    del vocab, vocab_index, idf, indptr
    gc.collect()
    mark_done("step3")
    backup(*[f"arrays/{n}" for n in PRODUCT_FILES])

PROD_META = json.loads((ARR_DIR / "prod_meta.json").read_text())
print(PROD_META)
ram("after step 3")

  lookup arrays equal length     OK
  attribute table rows           OK
  TF-IDF row pointers            OK
  TF-IDF index file complete     OK
  TF-IDF data file complete      OK
  vocabulary size                OK
Reusing Part 3's product tables.
Step 3 already completed - skipping.
{'n_prod': 11500314, 'duplicates': 0, 'non_integer_wid': 0, 'needed': 12223304, 'needed_missing': 722990, 'vocab': 368021, 'title_nnz': 268204961}
[RAM] after step 3: 0.36 GB in use


In [ ]:
LOOKUP_WID = np.load(ARR_DIR / "lookup_wid.npy", mmap_mode="r")
LOOKUP_ROW = np.load(ARR_DIR / "lookup_row.npy", mmap_mode="r")
PROD_ATTR = np.load(ARR_DIR / "prod_attr.npy", mmap_mode="r")
EMPTY_ROW = PROD_META["n_prod"]
TITLE_VOCAB = pd.read_parquet(ARR_DIR / "title_vocab.parquet")["term"]
TITLE_INDEX = pd.Index(TITLE_VOCAB)


def prod_rows(wids):
    # Row of each product ID in the product tables (EMPTY_ROW if the product has no metadata).
    wids = np.asarray(wids, dtype=np.int64)
    i = np.searchsorted(LOOKUP_WID, wids)
    i_c = np.minimum(i, len(LOOKUP_WID) - 1)
    found = np.asarray(LOOKUP_WID[i_c]) == wids
    return np.where(found, np.asarray(LOOKUP_ROW[i_c]), EMPTY_ROW)


def open_title_matrix():
    indptr = np.load(ARR_DIR / "title_indptr.npy", mmap_mode="r")
    nnz = int(indptr[-1])
    indices = np.memmap(ARR_DIR / "title_indices.bin", dtype=np.int32, mode="r", shape=(nnz,))
    data = np.memmap(ARR_DIR / "title_data.bin", dtype=np.float32, mode="r", shape=(nnz,))
    return sp.csr_matrix((data, indices, np.asarray(indptr)), shape=(len(indptr) - 1, len(TITLE_VOCAB)))


TITLES = open_title_matrix()
print("title matrix:", TITLES.shape, "non-zeros:", TITLES.nnz)

## Step 3 — Embeddings (item2vec and term2vec)

**What this step does.** It restores step_3_person_match_feature_engineering.ipynb's embeddings and checks them:
- the item2vec keys (sorted product IDs) must match the vectors in number and be sorted;
- the term2vec matrix must have one row per TF-IDF vocabulary term.

Anything missing or inconsistent is retrained with step_3_person_match_feature_engineering.ipynb's settings: word2vec skip-gram, 64 dimensions, window 5, 5 epochs. Item2vec is trained on the cleaned histories of all eligible sessions, products seen at least 5 times. Term2vec is trained on the product titles, which come from the title cache (restored from the backup, or rebuilt from the product file).

A retrained model is not bit-for-bit identical to step_3_person_match_feature_engineering.ipynb's, because gensim trains on several threads. The affected features would differ slightly from those evaluated in step_3_person_match_feature_engineering.ipynb, and Step 8 would show small AUC differences for them.

In [ ]:
from gensim.models import Word2Vec


class HistoryCorpus:
    # One sentence (list of product-ID strings) per session, streamed from disk (step_3_person_match_feature_engineering.ipynb code).
    def __iter__(self):
        for i in range(N_CHUNKS):
            h = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet", columns=["session_id", "wid"])
            sid = h["session_id"].to_numpy()
            toks = h["wid"].to_numpy().astype(str)
            cuts = np.flatnonzero(np.diff(sid)) + 1
            for sent in np.split(toks, cuts):
                if len(sent):
                    yield sent.tolist()


class TitleCorpus:
    # One sentence (list of term IDs) per product title, streamed from disk (step_3_person_match_feature_engineering.ipynb code).
    def __iter__(self):
        for f in sorted(TITLE_DIR.glob("titles_*.parquet")):
            for name in pd.read_parquet(f)["name"].to_numpy(dtype=object):
                if name:
                    yield name.split(TERM_SEP)


def i2v_ok():
    if not restore(*[f"arrays/{n}" for n in I2V_FILES]):
        return False
    try:
        k = np.load(ARR_DIR / "i2v_keys.npy", mmap_mode="r"); v = np.load(ARR_DIR / "i2v_vecs.npy", mmap_mode="r")
        return len(k) == v.shape[0] and v.shape[1] == EMB_DIM and bool(np.all(np.diff(k[:100_000]) > 0))
    except Exception:
        return False


def t2v_ok():
    if not restore(*[f"arrays/{n}" for n in T2V_FILES]):
        return False
    try:
        return np.load(ARR_DIR / "t2v_term_vecs.npy", mmap_mode="r").shape == (len(TITLE_VOCAB), EMB_DIM)
    except Exception:
        return False


def ensure_title_cache():
    # Title texts of the needed products, one Parquet file per product chunk (as written by step_3_person_match_feature_engineering.ipynb Step 3).
    if any(TITLE_DIR.glob("titles_*.parquet")) or restore("titles"):
        return
    print("Title cache not available - re-extracting titles from the product file.")
    needed = ensure_needed_wids()
    for i, ch in enumerate(product_chunks(["wid", "name"])):
        w = pd.to_numeric(ch["wid"], errors="coerce")
        keep = w.notna().to_numpy()
        keep[keep] = isin_sorted(w[keep].to_numpy(np.int64), needed)
        ch[keep][["name"]].fillna("").to_parquet(TITLE_DIR / f"titles_{i:03d}.parquet", index=False)


workers = max(1, os.cpu_count() or 1)
if i2v_ok():
    print("Reusing step_3_person_match_feature_engineering.ipynb's item2vec vectors.")
else:
    print("Item2vec vectors missing or inconsistent - retraining (step_3_person_match_feature_engineering.ipynb settings).")
    t0 = time.time()
    i2v = Word2Vec(HistoryCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=I2V_MIN_COUNT, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    keys = np.array(i2v.wv.index_to_key, dtype=np.int64)
    vecs = normalize(i2v.wv.vectors).astype(np.float32)
    order = np.argsort(keys)
    np.save(ARR_DIR / "i2v_keys.npy", keys[order]); np.save(ARR_DIR / "i2v_vecs.npy", vecs[order])
    print(f"item2vec: {len(keys):,} products with vectors ({time.time() - t0:.0f}s)")
    del i2v, keys, vecs, order; gc.collect()
    backup(*[f"arrays/{n}" for n in I2V_FILES])

if t2v_ok():
    print("Reusing step_3_person_match_feature_engineering.ipynb's term2vec vectors.")
else:
    print("Term2vec vectors missing or inconsistent - retraining (step_3_person_match_feature_engineering.ipynb settings).")
    ensure_title_cache()
    t0 = time.time()
    t2v = Word2Vec(TitleCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=TITLE_MIN_DF, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    E = np.zeros((len(TITLE_VOCAB), EMB_DIM), np.float32)
    k2i = t2v.wv.key_to_index
    pos = np.array([k2i.get(t, -1) for t in TITLE_VOCAB])
    E[pos >= 0] = t2v.wv.vectors[pos[pos >= 0]]
    np.save(ARR_DIR / "t2v_term_vecs.npy", E)
    print(f"term2vec: {(pos >= 0).sum():,} of {len(TITLE_VOCAB):,} title terms with vectors ({time.time() - t0:.0f}s)")
    del t2v, E, k2i, pos; gc.collect()
    backup(*[f"arrays/{n}" for n in T2V_FILES])
shutil.rmtree(TITLE_DIR, ignore_errors=True)    # the title texts are not needed beyond this point

I2V_KEYS = np.load(ARR_DIR / "i2v_keys.npy", mmap_mode="r")
I2V_VECS = np.load(ARR_DIR / "i2v_vecs.npy", mmap_mode="r")
T2V_TERMS = np.load(ARR_DIR / "t2v_term_vecs.npy", mmap_mode="r")


def i2v_rows(wids):
    # Index of each product's item2vec vector, or -1 if it has none (step_3_person_match_feature_engineering.ipynb code).
    wids = np.asarray(wids, dtype=np.int64)
    if len(I2V_KEYS) == 0:
        return np.full(len(wids), -1)
    i = np.minimum(np.searchsorted(I2V_KEYS, wids), len(I2V_KEYS) - 1)
    return np.where(np.asarray(I2V_KEYS[i]) == wids, i, -1)


print("item vectors:", I2V_VECS.shape, " term vectors:", T2V_TERMS.shape)
ram("after step 3")

## Step 4 — Training-set priors

**What this step does.** It reuses step_3_person_match_feature_engineering.ipynb's priors files when there is one per chunk with the same number of rows as the chunk's candidates. Otherwise it recomputes them with step_3_person_match_feature_engineering.ipynb's Step 5 code, unchanged: smoothed engagement rates (pos + 20·p₀) / (n + 20) from training sessions only, out-of-fold for training rows (5 folds), and log(1 + training impressions). The priors are stored in the chunks' row order, so they must match the session chunks of Step 1.

In [ ]:
def priors_ok():
    # Priors are reusable only if every chunk has a priors file with as many rows as its candidates.
    for i in range(N_CHUNKS):
        f = STAGE_DIR / f"priors_{i:03d}.parquet"
        if not f.exists() and not restore(f"stage/priors_{i:03d}.parquet"):
            print("Priors missing - recomputing."); return False
        if pq.ParquetFile(f).metadata.num_rows != pq.ParquetFile(STAGE_DIR / f"cand_{i:03d}.parquet").metadata.num_rows:
            print("Priors do not match the session chunks - recomputing."); return False
    print("Reusing step_3_person_match_feature_engineering.ipynb's training-set priors.")
    return True


if priors_ok():
    print("Step 5 already completed - skipping.")
else:
    parts = []
    for i in range(N_CHUNKS):
        c = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet", columns=["session_id", "wid", "label"])
        a = PROD_ATTR[prod_rows(c["wid"].to_numpy())]
        c["brand"] = a[:, BRAND]; c["cate3"] = a[:, CATE3]
        c["chunk"] = np.int16(i)
        parts.append(c)
    cand_all = pd.concat(parts, ignore_index=True); del parts
    split_of = sessions.set_index("session_id")["split"]
    cand_all["split"] = pd.Categorical(split_of.reindex(cand_all["session_id"].to_numpy()).to_numpy())
    cand_all["pos"] = (cand_all["label"] > 0).astype(np.int8)

    # folds over training sessions
    tr_sess = sessions.loc[sessions["split"] == "train", "session_id"].to_numpy()
    fold_of = pd.Series(np.random.default_rng(SEED).permutation(len(tr_sess)) % N_FOLDS, index=tr_sess)
    is_tr = (cand_all["split"] == "train").to_numpy()
    cand_all["fold"] = np.int8(-1)
    cand_all.loc[is_tr, "fold"] = fold_of.reindex(cand_all.loc[is_tr, "session_id"].to_numpy()).to_numpy()
    tr = cand_all[is_tr]
    p0 = tr["pos"].mean()
    print(f"training rows: {len(tr):,}; positive rate p0 = {p0:.4f}")

    for key, col in [("wid", "item_ctr_train"), ("brand", "brand_ctr_train"), ("cate3", "cate3_ctr_train")]:
        full = tr.groupby(key)["pos"].agg(["size", "sum"])
        by_fold = tr.groupby([key, "fold"])["pos"].agg(["size", "sum"])
        keys = cand_all[key].to_numpy()
        n_full = full["size"].reindex(keys).fillna(0).to_numpy()
        p_full = full["sum"].reindex(keys).fillna(0).to_numpy()
        idx = pd.MultiIndex.from_arrays([keys[is_tr], cand_all.loc[is_tr, "fold"].to_numpy()])
        n_full[is_tr] -= by_fold["size"].reindex(idx).fillna(0).to_numpy()   # leave own fold out
        p_full[is_tr] -= by_fold["sum"].reindex(idx).fillna(0).to_numpy()
        rate = (p_full + PRIOR_ALPHA * p0) / (n_full + PRIOR_ALPHA)
        if key != "wid":
            rate = np.where(keys == -1, np.nan, rate)
        cand_all[col] = rate.astype(np.float32)
        if key == "wid":
            cand_all["item_impr_train"] = np.log1p(full["size"].reindex(keys).fillna(0).to_numpy()).astype(np.float32)
        del full, by_fold, keys, n_full, p_full, idx, rate
    for i, g in cand_all.groupby("chunk", sort=True):
        g[["session_id", "wid"] + FAMILIES["G"]].to_parquet(STAGE_DIR / f"priors_{i:03d}.parquet", index=False)
    del cand_all, tr, fold_of
    gc.collect()
    mark_done("step5")
    backup(*[f"stage/priors_{i:03d}.parquet" for i in range(N_CHUNKS)])
ram("after step 5")

Reusing Part 3's training-set priors.
Step 5 already completed - skipping.
[RAM] after step 5: 0.47 GB in use


## Step 5 — Feature computation (step_3_person_match_feature_engineering.ipynb code)

The next cell defines `build_chunk_features`, copied unchanged from step_3_person_match_feature_engineering.ipynb. For one chunk of 2,000 sessions, it computes all 90 features. The definitions are in the proposal and the feature reference workbook. In short:
- every past action carries the weight w = type weight × exp(−elapsed days / 30), with CLICK 1, FLW 2, CART 3 and ORD 5;
- the families are computed in turn: context, A, B, C, D, E, F and G; family H comes last, because it is defined relative to the other candidates of the same slate;
- a missing value (`NaN`) means "no evidence" and is not replaced by zero.

In [ ]:
def group_bounds(sorted_local, n_groups):
    # Start and end row of each group in an array sorted by group.
    starts = np.searchsorted(sorted_local, np.arange(n_groups), side="left")
    ends = np.searchsorted(sorted_local, np.arange(n_groups), side="right")
    return starts, ends


def rowdot(A, B):
    # Row-wise dot product of two sparse matrices with the same shape.
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()


def nan_if(values, mask):
    out = np.asarray(values, dtype=np.float64).copy()
    out[mask] = np.nan
    return out


def keyed_merge(c_keys, h_keys, h_values, how="sum"):
    # For each candidate key (session, value), aggregate h_values over history rows with the same key.
    # Keys are 2-column int64 arrays; returns NaN where no history row matches.
    h = pd.DataFrame({"s": h_keys[0], "v": h_keys[1], "x": h_values})
    agg = h.groupby(["s", "v"])["x"].agg(how)
    idx = pd.MultiIndex.from_arrays([c_keys[0], c_keys[1]])
    return agg.reindex(idx).to_numpy(dtype=np.float64)


def build_chunk_features(i):
    cand = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet")
    hist = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet")
    pri = pd.read_parquet(STAGE_DIR / f"priors_{i:03d}.parquet")
    assert (pri["session_id"].to_numpy() == cand["session_id"].to_numpy()).all()
    assert (pri["wid"].to_numpy() == cand["wid"].to_numpy()).all()
    sess = sessions[sessions["chunk"] == i].sort_values("session_id")
    sids = sess["session_id"].to_numpy()
    nS = len(sids)
    c_loc = np.searchsorted(sids, cand["session_id"].to_numpy())
    h_loc = np.searchsorted(sids, hist["session_id"].to_numpy())
    nC, nH = len(cand), len(hist)
    F = {}

    c_wid, h_wid = cand["wid"].to_numpy(), hist["wid"].to_numpy()
    c_row, h_row = prod_rows(c_wid), prod_rows(h_wid)
    c_attr, h_attr = np.asarray(PROD_ATTR[c_row]), np.asarray(PROD_ATTR[h_row])
    typ = hist["type"].to_numpy().astype(np.int64)
    days = hist["days"].to_numpy().astype(np.float64)
    w = np.where(typ >= 0, TYPE_WEIGHT[np.clip(typ, 0, 3)], 1.0) * np.exp(-days / TAU_DAYS)
    tot_w = np.bincount(h_loc, weights=w, minlength=nS)
    h_len = np.bincount(h_loc, minlength=nS)
    h_start, h_end = group_bounds(h_loc, nS)
    c_start, c_end = group_bounds(c_loc, nS)
    has_h = h_len > 0
    last_idx = np.where(has_h, h_end - 1, 0)

    # ---------------- context (session level, broadcast to candidates) ----------------
    ctx = {}
    for t, tname in enumerate(TYPE_NAMES):
        for (a, b), wname in zip([(0, 7), (7, 90), (90, 365)], ["0_7d", "7_90d", "90_365d"]):
            m = (typ == t) & (days >= a) & (days < b)
            ctx[f"{tname}_count_{wname}"] = np.bincount(h_loc[m], minlength=nS)
    ctx["history_len_clean"] = h_len
    dser = pd.Series(days).groupby(h_loc)
    ctx["history_span_days"] = dser.max().reindex(range(nS)).to_numpy()
    ctx["days_since_last_interaction"] = dser.min().reindex(range(nS)).to_numpy()
    for t, tname in enumerate(TYPE_NAMES):
        ctx[f"share_{tname}"] = nan_if(np.bincount(h_loc[typ == t], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    c3 = h_attr[:, CATE3]
    m3 = c3 != -1
    cc = pd.DataFrame({"s": h_loc[m3], "v": c3[m3]}).value_counts()
    p = cc / cc.groupby(level=0).transform("sum")
    ent = (-(p * np.log(p))).groupby(level=0).sum()
    ctx["cate3_entropy"] = nan_if(ent.reindex(range(nS)).fillna(0).to_numpy(), ~has_h)
    br = h_attr[:, BRAND]
    ctx["n_distinct_brands"] = pd.DataFrame({"s": h_loc[br != -1], "v": br[br != -1]}).drop_duplicates()["s"] \
        .value_counts().reindex(range(nS)).fillna(0).to_numpy()
    qryless = (hist["hqry"].to_numpy(dtype=object) == "-1")
    ctx["qryless_share"] = nan_if(np.bincount(h_loc[qryless], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    same_next = np.zeros(nH, bool); same_next[:-1] = h_loc[:-1] == h_loc[1:]
    gap_next = np.zeros(nH); gap_next[:-1] = days[:-1] - days[1:]
    brk = same_next & (gap_next >= BURST_GAP_DAYS)
    hpos = np.arange(nH) - h_start[h_loc]
    last_brk = np.full(nS, -1); np.maximum.at(last_brk, h_loc[brk], hpos[brk])
    ctx["burst_len"] = h_len - (last_brk + 1)
    for k in FAMILIES["context"]:
        F[k] = np.asarray(ctx[k], dtype=np.float64)[c_loc]

    # ---------------- A. exact product (step_1_data_preprocessing.ipynb) ----------------
    defs = [("exact_wid_click_0_7d", 0, 0, 7), ("exact_wid_cart_0_7d", 1, 0, 7), ("exact_wid_ord_0_7d", 2, 0, 7),
            ("exact_wid_cart_7_90d", 1, 7, 90), ("exact_wid_ord_7_90d", 2, 7, 90), ("exact_wid_ord_90_365d", 2, 90, 365)]
    hdf = pd.DataFrame({"s": h_loc, "v": h_wid, "d": days})
    for name, t, a, b in defs:
        hdf[name] = ((typ == t) & (days >= a) & (days < b)).astype(np.int32)
    g = hdf.groupby(["s", "v"]).agg({**{n: "sum" for n, *_ in defs}, "d": "min"})
    idx = pd.MultiIndex.from_arrays([c_loc, c_wid])
    ga = g.reindex(idx)
    for name, *_ in defs:
        F[name] = ga[name].fillna(0).to_numpy()
    F["exact_wid_last_seen_days"] = ga["d"].to_numpy()
    del hdf, g, ga

    # ---------------- B. attribute affinity ----------------
    is_ord = (typ == TYPE_CODE["ORD"]).astype(np.float64)
    for j, k in enumerate(ATTRS):
        hv, cv = h_attr[:, j], c_attr[:, j]
        hm = hv != -1
        keys_h = (h_loc[hm], hv[hm])
        keys_c = (c_loc, cv)
        aff = keyed_merge(keys_c, keys_h, w[hm], "sum")
        last = keyed_merge(keys_c, keys_h, days[hm], "min")
        ordc = keyed_merge(keys_c, keys_h, is_ord[hm], "sum")
        miss = cv == -1
        F[f"{k}_aff"] = nan_if(aff, miss)
        F[f"{k}_share"] = nan_if(aff / np.maximum(tot_w[c_loc], 1e-12), miss)
        F[f"{k}_last_seen_days"] = nan_if(last, miss)
        F[f"{k}_ord_cnt"] = nan_if(ordc, miss)

    # ---------------- C. title similarity ----------------
    Xc, Xh = TITLES[c_row], TITLES[h_row]
    c_nnz = np.diff(Xc.indptr)
    S = sp.csr_matrix((w, (h_loc, np.arange(nH))), shape=(nS, nH))
    P = normalize(S @ Xh)
    p_nnz = np.diff(P.indptr)
    F["title_cos_profile"] = nan_if(rowdot(P[c_loc], Xc), (c_nnz == 0) | (p_nnz[c_loc] == 0))
    S_ord = sp.csr_matrix((w * is_ord, (h_loc, np.arange(nH))), shape=(nS, nH))
    P_ord = normalize(S_ord @ Xh)
    po_nnz = np.diff(P_ord.indptr)
    F["title_cos_profile_ord"] = nan_if(rowdot(P_ord[c_loc], Xc), (c_nnz == 0) | (po_nnz[c_loc] == 0))
    Xlast = Xh[last_idx]
    l_nnz = np.diff(Xlast.indptr)
    F["last_title_cos"] = nan_if(rowdot(Xlast[c_loc], Xc), (c_nnz == 0) | (l_nnz[c_loc] == 0) | ~has_h[c_loc])
    # overlap of candidate title terms with the user's past query terms
    hq = hist["hqry"].to_numpy(dtype=object)
    qm = ~qryless
    q_toks, q_lens = split_flat(pd.Series(hq[qm]), TERM_SEP)
    q_cols = TITLE_INDEX.get_indexer(q_toks)
    q_rows = np.repeat(h_loc[qm], q_lens)
    ok = q_cols >= 0
    Q = sp.csr_matrix((np.ones(ok.sum(), np.float32), (q_rows[ok], q_cols[ok])), shape=(nS, len(TITLE_VOCAB)))
    Q.data[:] = 1.0                                            # term present (duplicates collapsed)
    Bc = Xc.copy(); Bc.data[:] = 1.0
    F["title_hist_qry_overlap"] = nan_if(rowdot(Q[c_loc], Bc) / np.maximum(c_nnz, 1), c_nnz == 0)

    # ---------------- D. embedding similarity ----------------
    c_vi, h_vi = i2v_rows(c_wid), i2v_rows(h_wid)
    Vc = np.zeros((nC, EMB_DIM), np.float32); Vc[c_vi >= 0] = I2V_VECS[c_vi[c_vi >= 0]]
    hk = h_vi >= 0
    Vh = np.zeros((nH, EMB_DIM), np.float32); Vh[hk] = I2V_VECS[h_vi[hk]]
    n_known = np.bincount(h_loc[hk], minlength=nS)

    def session_sum(mask, M):
        # Sum the rows of M selected by mask within each session (a sparse indicator product, no Python loop).
        A = sp.csr_matrix((np.ones(mask.sum()), (h_loc[mask], np.flatnonzero(mask))), shape=(nS, nH))
        return np.asarray(A @ M)

    mean_v = session_sum(hk, Vh) / np.maximum(n_known, 1)[:, None]
    c_unknown = c_vi < 0
    F["i2v_cos_mean"] = nan_if((Vc * mean_v[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    cum = np.cumsum(hk)
    within = cum - np.r_[0, cum][h_start][h_loc]              # known vectors up to and including this row
    rank_from_end = n_known[h_loc] - within + hk               # 1 = most recent known vector of the session
    last5 = hk & (rank_from_end <= 5)
    l5 = normalize(session_sum(last5, Vh))
    F["i2v_cos_last5"] = nan_if((Vc * l5[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    F["last_i2v_cos"] = nan_if((Vc * Vh[last_idx][c_loc]).sum(1), c_unknown | ~has_h[c_loc] | ~hk[last_idx][c_loc])
    Tc = normalize(np.asarray(Xc @ T2V_TERMS)); Th = normalize(np.asarray(Xh @ T2V_TERMS))
    th_ok = np.abs(Th).sum(1) > 0
    tprof = normalize(session_sum(th_ok, Th))
    t_nn = np.abs(tprof).sum(1) > 0
    F["t2v_cos_profile"] = nan_if((Tc * tprof[c_loc]).sum(1), (np.abs(Tc).sum(1) == 0) | ~t_nn[c_loc])

    # maximum similarity with any single history item (loop over sessions)
    tmax = np.full(nC, np.nan); imax = np.full(nC, np.nan)
    for s in range(nS):
        a, b, c0, d0 = c_start[s], c_end[s], h_start[s], h_end[s]
        if b == a or d0 == c0:
            continue
        M = (Xc[a:b] @ Xh[c0:d0].T).toarray()
        tmax[a:b] = M.max(axis=1)
        kh = hk[c0:d0]
        if kh.any():
            imax[a:b] = (Vc[a:b] @ Vh[c0:d0][kh].T).max(axis=1)
    F["title_cos_max"] = nan_if(tmax, c_nnz == 0)
    F["i2v_cos_max"] = nan_if(imax, c_unknown)

    # ---------------- E. recency of interest ----------------
    for name, j in [("last_same_brand", BRAND), ("last_same_cate3", CATE3)]:
        lv = h_attr[last_idx, j][c_loc]
        cv = c_attr[:, j]
        F[name] = nan_if((cv == lv).astype(np.float64), (cv == -1) | (lv == -1) | ~has_h[c_loc])

    # ---------------- F. query-conditioned history ----------------
    q_terms = sess["query"].to_numpy(dtype=object)
    qt, ql = split_flat(pd.Series(q_terms), TERM_SEP)
    query_long = pd.DataFrame({"s": np.repeat(np.arange(nS), ql), "term": qt}).drop_duplicates()
    hq_long = pd.DataFrame({"r": np.repeat(np.flatnonzero(qm), q_lens), "s": q_rows, "term": q_toks})
    matched_rows = hq_long.merge(query_long, on=["s", "term"])["r"].unique()
    mt = np.zeros(nH, bool); mt[matched_rows] = True
    F["qhist_n_actions"] = np.bincount(h_loc[mt], minlength=nS)[c_loc].astype(np.float64)
    F["qhist_wid_cnt"] = np.nan_to_num(keyed_merge((c_loc, c_wid), (h_loc[mt], h_wid[mt]), np.ones(mt.sum()), "sum"))
    for name, j in [("qhist_brand_aff", BRAND), ("qhist_cate3_aff", CATE3)]:
        hv = h_attr[:, j]; mm = mt & (hv != -1)
        F[name] = nan_if(keyed_merge((c_loc, c_attr[:, j]), (h_loc[mm], hv[mm]), w[mm], "sum"), c_attr[:, j] == -1)

    # ---------------- G. training-set priors ----------------
    for k in FAMILIES["G"]:
        F[k] = pri[k].to_numpy(dtype=np.float64)

    # ---------------- H. position within the slate ----------------
    for f in H_BASE:
        x = pd.Series(F[f])
        grp = x.groupby(c_loc)
        F[f + "_slate_pct"] = grp.rank(ascending=False, pct=True).to_numpy()
        sd = grp.transform("std").replace(0, np.nan)
        F[f + "_slate_z"] = ((x - grp.transform("mean")) / sd).to_numpy()
    slate = np.bincount(c_loc, minlength=nS)
    F["slate_size"] = slate[c_loc].astype(np.float64)
    cv3 = c_attr[:, CATE3]
    same3 = pd.Series(1, index=pd.MultiIndex.from_arrays([c_loc, cv3])).groupby(level=[0, 1]).transform("sum").to_numpy()
    F["cate3_slate_share"] = nan_if(same3 / slate[c_loc], cv3 == -1)

    # ---------------- assemble ----------------
    out = pd.DataFrame({
        "session_id": cand["session_id"].to_numpy(),
        "wid": c_wid,
        "label": cand["label"].to_numpy(),
        "split": sess["split"].to_numpy()[c_loc],
    })
    feats = pd.DataFrame({k: np.asarray(F[k], dtype=np.float32) for k in FEATURE_COLUMNS})
    return pd.concat([out, feats], axis=1)

## Step 6 — Compute all chunks

**What this step does.** It runs `build_chunk_features` for each of the 86 chunks and writes `chunks/feat_XXX.parquet` to Drive. A chunk that already exists is skipped, so after a runtime limit, rerunning the notebook from the top continues where it stopped. If the final split files are already complete (`MERGE_COMPLETE.json` exists), this step is skipped entirely.

In [18]:
if MERGE_DONE.exists():
    print("Final split files already complete - nothing to compute.")
else:
    t_start = time.time()
    for i in range(N_CHUNKS):
        f = CHUNK_DIR / f"feat_{i:03d}.parquet"
        if f.exists():
            continue
        t0 = time.time()
        feats = build_chunk_features(i)
        assert list(feats.columns) == ID_COLUMNS + FEATURE_COLUMNS
        feats.to_parquet(f, index=False, compression="zstd")
        print(f"chunk {i + 1}/{N_CHUNKS}: {len(feats):,} rows in {time.time() - t0:.0f}s "
              f"(elapsed {(time.time() - t_start) / 60:.1f} min)")
        del feats; gc.collect()
        if i % 10 == 0:
            ram(f"after chunk {i}")
    print("chunks on Drive:", len(list(CHUNK_DIR.glob("feat_*.parquet"))), "of", N_CHUNKS)

chunk 1/86: 179,397 rows in 11s (elapsed 0.2 min)
[RAM] after chunk 0: 3.92 GB in use
chunk 2/86: 183,012 rows in 11s (elapsed 0.4 min)
chunk 3/86: 176,513 rows in 11s (elapsed 0.6 min)
chunk 4/86: 179,332 rows in 11s (elapsed 0.8 min)
chunk 5/86: 178,637 rows in 11s (elapsed 0.9 min)
chunk 6/86: 177,387 rows in 11s (elapsed 1.1 min)
chunk 7/86: 177,790 rows in 11s (elapsed 1.3 min)
chunk 8/86: 180,005 rows in 11s (elapsed 1.5 min)
chunk 9/86: 177,530 rows in 10s (elapsed 1.7 min)
chunk 10/86: 177,475 rows in 11s (elapsed 1.8 min)
chunk 11/86: 178,821 rows in 11s (elapsed 2.0 min)
[RAM] after chunk 10: 4.04 GB in use
chunk 12/86: 177,772 rows in 10s (elapsed 2.2 min)
chunk 13/86: 182,592 rows in 11s (elapsed 2.4 min)
chunk 14/86: 181,554 rows in 11s (elapsed 2.6 min)
chunk 15/86: 176,980 rows in 10s (elapsed 2.7 min)
chunk 16/86: 177,250 rows in 11s (elapsed 2.9 min)
chunk 17/86: 181,853 rows in 11s (elapsed 3.1 min)
chunk 18/86: 178,183 rows in 11s (elapsed 3.3 min)
chunk 19/86: 179,3

## Step 7 — Write one file per split (storage option A)

**What this step does.** It streams the chunk files, in order, into three Parquet files: `train.parquet`, `val.parquet` and `test.parquet`. Each chunk is appended to the three files as a separate row group, and then, if `DELETE_EACH_CHUNK_AFTER_MERGE` is `True`, deleted immediately. On Drive, the data therefore exist only once plus one chunk at any time. The group sizes (candidates per session) are collected along the way and saved as `*_groups.npy`, together with `feature_list.json` and the feature dictionary. `MERGE_COMPLETE.json` is written last.

**Why this layout suits LightGBM.** LightGBM's ranking objective needs the rows of each query (here, each session) to be contiguous, plus the number of rows per query, its `group` argument. Each chunk keeps its sessions' candidates together and in session order, and filtering a chunk by split keeps whole sessions. The rows of every split file are therefore already grouped by session, and the group sizes can be read off directly.

**If the merge is interrupted.** The partly written split files are incomplete, and the chunks already merged have been deleted. Rerun the notebook from the top: Step 6 recomputes the deleted chunks, and this step starts the split files afresh. Nothing is lost except the time to recompute those chunks.

In [19]:
if MERGE_DONE.exists():
    print("Split files already complete:", json.loads(MERGE_DONE.read_text()))
else:
    chunk_files = [CHUNK_DIR / f"feat_{i:03d}.parquet" for i in range(N_CHUNKS)]
    missing = [f.name for f in chunk_files if not f.exists()]
    assert not missing, f"chunks missing ({len(missing)}): rerun Step 6 first"
    for s in ["train", "val", "test"]:
        (OUT_DIR / f"{s}.parquet").unlink(missing_ok=True)        # start the split files afresh
    writers, groups, counts = {}, {s: [] for s in ["train", "val", "test"]}, Counter()
    for f in chunk_files:
        df = pd.read_parquet(f)
        for s, part in df.groupby("split", sort=False):
            part = part.drop(columns="split")
            sid = part["session_id"].to_numpy()
            assert (np.diff(sid) >= 0).all(), "rows must be sorted by session"
            cuts = np.flatnonzero(np.diff(sid)) + 1
            groups[s].append(np.diff(np.r_[0, cuts, len(sid)]))
            table = pa.Table.from_pandas(part, preserve_index=False)
            if s not in writers:
                writers[s] = pq.ParquetWriter(OUT_DIR / f"{s}.parquet", table.schema, compression="zstd")
            writers[s].write_table(table)
            counts[s] += len(part)
        del df
        if DELETE_EACH_CHUNK_AFTER_MERGE:
            f.unlink()
    summary = {}
    for s, wtr in writers.items():
        wtr.close()
        g = np.concatenate(groups[s]).astype(np.int32)
        np.save(OUT_DIR / f"{s}_groups.npy", g)
        assert g.sum() == counts[s]
        summary[s] = {"rows": int(counts[s]), "sessions": int(len(g)),
                      "GB": round((OUT_DIR / f"{s}.parquet").stat().st_size / 1e9, 3)}
    (OUT_DIR / "feature_list.json").write_text(json.dumps(FEATURE_COLUMNS, indent=1))
    MERGE_DONE.write_text(json.dumps(summary, indent=1))
    if DELETE_EACH_CHUNK_AFTER_MERGE:
        shutil.rmtree(CHUNK_DIR, ignore_errors=True)
    print(json.dumps(summary, indent=1))
    print("total rows:", sum(counts.values()), f"(expected {EXPECTED['candidate_rows']:,} on the full data)")

{
 "val": {
  "rows": 1536651,
  "sessions": 17117,
  "GB": 0.112
 },
 "train": {
  "rows": 12279873,
  "sessions": 136940,
  "GB": 0.885
 },
 "test": {
  "rows": 1531525,
  "sessions": 17118,
  "GB": 0.111
 }
}
total rows: 15348049 (expected 15,348,049 on the full data)


The feature dictionary (step_3_person_match_feature_engineering.ipynb code, unchanged) is saved next to the split files as `feature_dictionary_part3.csv`.

In [ ]:
DESCR = {
    "exact_wid_click_0_7d": "Clicks on this exact product 0-7 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_cart_0_7d": "Cart additions of this exact product 0-7 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_ord_0_7d": "Orders of this exact product 0-7 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_cart_7_90d": "Cart additions of this exact product 7-90 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_ord_7_90d": "Orders of this exact product 7-90 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_ord_90_365d": "Orders of this exact product 90-365 days before the query (step_1_data_preprocessing.ipynb)",
    "exact_wid_last_seen_days": "Days since the most recent action on this exact product; missing if never seen (step_1_data_preprocessing.ipynb)",
    "title_cos_profile": "Cosine of the candidate's TF-IDF title with the user's weighted title profile",
    "title_cos_profile_ord": "As title_cos_profile, using ordered products only; missing if no orders",
    "title_cos_max": "Largest title cosine with any single history product",
    "title_hist_qry_overlap": "Share of the candidate's title terms that occur in the user's past queries",
    "i2v_cos_mean": "Mean item2vec cosine with the history products that have vectors",
    "i2v_cos_max": "Largest item2vec cosine with any history product",
    "i2v_cos_last5": "Item2vec cosine with the mean of the five most recent history vectors",
    "t2v_cos_profile": "Cosine of the candidate's term2vec title vector with the mean history title vector",
    "last_same_brand": "1 if the candidate has the brand of the most recent action, else 0",
    "last_same_cate3": "1 if the candidate has the level-3 category of the most recent action, else 0",
    "last_title_cos": "Title cosine with the product of the most recent action",
    "last_i2v_cos": "Item2vec cosine with the product of the most recent action",
    "qhist_n_actions": "Number of history actions whose past query shares a term with the test query (session level)",
    "qhist_wid_cnt": "Number of those query-matched actions on the candidate's own product",
    "qhist_brand_aff": "Weighted affinity for the candidate's brand among query-matched actions",
    "qhist_cate3_aff": "Weighted affinity for the candidate's level-3 category among query-matched actions",
    "item_ctr_train": "Smoothed engagement rate of the product in training sessions (out-of-fold for training rows)",
    "brand_ctr_train": "Smoothed engagement rate of the brand in training sessions (out-of-fold for training rows)",
    "cate3_ctr_train": "Smoothed engagement rate of the level-3 category in training sessions (out-of-fold)",
    "item_impr_train": "log(1 + number of training rows showing the product)",
    "cate3_slate_share": "Share of the slate's candidates with the candidate's level-3 category",
    "slate_size": "Number of candidates in the session",
    "history_len_clean": "Number of history actions after cleaning (step_1_data_preprocessing.ipynb)",
    "history_span_days": "Days since the oldest history action (step_1_data_preprocessing.ipynb)",
    "days_since_last_interaction": "Days since the most recent history action (step_1_data_preprocessing.ipynb)",
    "cate3_entropy": "Entropy of the level-3 categories in the history (low = focused user)",
    "n_distinct_brands": "Number of distinct brands in the history",
    "qryless_share": "Share of history actions not preceded by a query (-1)",
    "burst_len": "Number of actions in the final run of actions less than 30 minutes apart",
}
for k, kn in zip(ATTRS, ["brand", "shop", "level-1 category", "level-2 category", "level-3 category", "level-4 category"]):
    DESCR[f"{k}_aff"] = f"Recency- and action-weighted sum of history actions with the candidate's {kn}"
    DESCR[f"{k}_share"] = f"{k}_aff as a share of the session's total history weight"
    DESCR[f"{k}_last_seen_days"] = f"Days since the candidate's {kn} last occurred in the history"
    DESCR[f"{k}_ord_cnt"] = f"Number of orders of products with the candidate's {kn}"
for t in TYPE_NAMES:
    DESCR[f"share_{t}"] = f"Share of history actions of type {t.upper()}"
    for wname in ["0_7d", "7_90d", "90_365d"]:
        DESCR[f"{t}_count_{wname}"] = f"Number of {t.upper()} actions {wname.replace('_', '-').replace('d', ' days')} before the query (step_1_data_preprocessing.ipynb)"
for f in H_BASE:
    DESCR[f + "_slate_pct"] = f"Descending percentile rank of {f} among the session's candidates (highest value = 1/n, lowest = 1)"
    DESCR[f + "_slate_z"] = f"z-score of {f} among the session's candidates"

def missing_meaning(f):
    fam = FAMILY_OF[f]
    if f.startswith("exact_wid") and f != "exact_wid_last_seen_days" or f in ("qhist_wid_cnt", "qhist_n_actions", "slate_size"):
        return "never missing (0 = none)"
    if fam == "context":
        return "missing only if the session has no history"
    if fam == "G":
        return "missing if the brand or category is unknown"
    if fam == "H":
        return "missing if the underlying feature is missing or the slate has one candidate"
    return "missing = no evidence (value never seen in the history, or no title / vector)"

assert set(DESCR) == set(FEATURE_COLUMNS), set(FEATURE_COLUMNS) ^ set(DESCR)
feature_dictionary = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "family": [FAMILY_OF[f] for f in FEATURE_COLUMNS],
    "description": [DESCR[f] for f in FEATURE_COLUMNS],
    "missing_value": [missing_meaning(f) for f in FEATURE_COLUMNS],
})
feature_dictionary.to_csv(P3_DIR / "feature_dictionary_part3.csv", index=False)
print("Saved", P3_DIR / "feature_dictionary_part3.csv")
display(feature_dictionary.head(12))

Saved /content/drive/MyDrive/BT4222 Final/JDSearch/processed/lgbm_features_all/feature_dictionary_part3.csv


,feature,family,description,missing_value
0,exact_wid_click_0_7d,A,Clicks on this exact product 0-7 days before t...,never missing (0 = none)
1,exact_wid_cart_0_7d,A,Cart additions of this exact product 0-7 days ...,never missing (0 = none)
2,exact_wid_ord_0_7d,A,Orders of this exact product 0-7 days before t...,never missing (0 = none)
3,exact_wid_cart_7_90d,A,Cart additions of this exact product 7-90 days...,never missing (0 = none)
4,exact_wid_ord_7_90d,A,Orders of this exact product 7-90 days before ...,never missing (0 = none)
5,exact_wid_ord_90_365d,A,Orders of this exact product 90-365 days befor...,never missing (0 = none)
6,exact_wid_last_seen_days,A,Days since the most recent action on this exac...,missing = no evidence (value never seen in the...
7,brand_aff,B,Recency- and action-weighted sum of history ac...,missing = no evidence (value never seen in the...
8,brand_share,B,brand_aff as a share of the session's total hi...,missing = no evidence (value never seen in the...
9,brand_last_seen_days,B,Days since the candidate's brand last occurred...,missing = no evidence (value never seen in the...


## Step 8 — Verification against the step_3_person_match_feature_engineering.ipynb feature evaluation

**What this step does.** It recomputes the within-session AUC of every feature on step_3_person_match_feature_engineering.ipynb's evaluation sample: 10,000 training sessions drawn with seed 42, using the same definition. For the 31 features whose step_3_person_match_feature_engineering.ipynb values are recorded in the workbook, it shows the difference. Agreement to the three decimals reported confirms that the features were reproduced exactly; if item2vec or term2vec was retrained in Step 3, those features may differ slightly. The table also gives each feature's coverage, the share of rows where it is not missing.

In [ ]:
PART3_AUC = {"item_impr_train": 0.760, "i2v_cos_max_slate_pct": 0.695, "i2v_cos_max": 0.679, "i2v_cos_max_slate_z": 0.676,
             "i2v_cos_last5": 0.673, "i2v_cos_mean": 0.662, "title_cos_max": 0.642, "title_cos_max_slate_z": 0.642,
             "last_i2v_cos": 0.642, "title_cos_max_slate_pct": 0.360, "title_cos_profile": 0.619,
             "title_cos_profile_slate_z": 0.619, "title_cos_profile_slate_pct": 0.383, "brand_ctr_train": 0.587,
             "shop_share": 0.586, "shop_aff": 0.586, "last_title_cos": 0.586, "shop_ord_cnt": 0.585,
             "shop_aff_slate_pct": 0.583, "shop_last_seen_days": 0.583, "exact_wid_last_seen_days": 0.555,
             "brand_aff": 0.553, "brand_share": 0.553, "title_cos_profile_ord": 0.552, "brand_ord_cnt": 0.551,
             "shop_aff_slate_z": 0.551, "item_ctr_train_slate_z": 0.550, "item_ctr_train": 0.550,
             "item_ctr_train_slate_pct": 0.450, "cate3_slate_share": 0.550, "qhist_wid_cnt": 0.527}


def session_auc(session_id, x, engaged):
    # Mean over sessions of the within-session AUC (ties = 1/2, missing values rank lowest) - step_3_person_match_feature_engineering.ipynb code.
    x = np.where(np.isnan(x), -np.inf, x)
    r = pd.Series(x).groupby(session_id).rank(method="average").to_numpy()
    g = pd.DataFrame({"s": session_id, "r": r * engaged, "p": engaged, "n": ~engaged}).groupby("s")[["r", "p", "n"]].sum()
    g = g[(g["p"] > 0) & (g["n"] > 0)]
    return ((g["r"] - g["p"] * (g["p"] + 1) / 2) / (g["p"] * g["n"])).mean()


rng = np.random.default_rng(SEED)
train_sessions = sessions.loc[sessions["split"] == "train", "session_id"].to_numpy()
sample_ids = np.sort(rng.choice(train_sessions, size=min(SCREEN_SESSIONS, len(train_sessions)), replace=False))
sample = pq.read_table(OUT_DIR / "train.parquet", filters=[("session_id", "in", sample_ids.tolist())]).to_pandas()
sid, eng = sample["session_id"].to_numpy(), (sample["label"] > 0).to_numpy()
check = pd.DataFrame({
    "family": [FAMILY_OF[f] for f in FEATURE_COLUMNS],
    "step_3_person_match_feature_engineering.ipynb AUC": pd.Series(PART3_AUC).reindex(FEATURE_COLUMNS).to_numpy(),
    "reproduced AUC": [session_auc(sid, sample[f].to_numpy(np.float64), eng) for f in FEATURE_COLUMNS],
    "coverage": sample[FEATURE_COLUMNS].notna().mean().to_numpy(),
}, index=FEATURE_COLUMNS)
check["difference"] = check["reproduced AUC"] - check["step_3_person_match_feature_engineering.ipynb AUC"]
pd.set_option("display.max_rows", 100)
display(check.round(4))
print(f"Sample: {len(sample_ids):,} training sessions, {len(sample):,} rows. Largest |difference| "
      f"(31 recorded features): {check['difference'].abs().max():.4f} (step_3_person_match_feature_engineering.ipynb values rounded to 3 decimals)")
del sample

,family,Part 3 AUC,reproduced AUC,coverage,difference
exact_wid_click_0_7d,A,NaN,0.5400,1.0000,NaN
exact_wid_cart_0_7d,A,NaN,0.5146,1.0000,NaN
exact_wid_ord_0_7d,A,NaN,0.5020,1.0000,NaN
exact_wid_cart_7_90d,A,NaN,0.5108,1.0000,NaN
exact_wid_ord_7_90d,A,NaN,0.5031,1.0000,NaN
exact_wid_ord_90_365d,A,NaN,0.5042,1.0000,NaN
exact_wid_last_seen_days,A,0.555,0.5547,0.0077,-0.0003
brand_aff,B,0.553,0.5530,0.2192,-0.0000
brand_share,B,0.553,0.5530,0.2192,-0.0000
brand_last_seen_days,B,NaN,0.5485,0.2192,NaN


Sample: 10,000 training sessions, 890,023 rows. Largest |difference| (31 recorded features): 0.0005 (Part 3 values rounded to 3 decimals)


## Step 9 — Feature loader: choose the features and load them for training

This section turns the saved files into the in-memory data that LightGBM trains on. **It can be run on its own** in a fresh session (after mounting Drive in Step 0's first cell), without rerunning Steps 1 to 8.

### Which files are needed

For each split you want to load, the loader reads three files from `processed/lgbm_features_all/`:

| File | Why it is needed |
|---|---|
| `{split}.parquet` | The data. Only the columns you select are read, because Parquet stores each column separately. |
| `{split}_groups.npy` | The number of candidates per session, in row order: LightGBM's `group` argument. Without it, LightGBM cannot tell where one session ends and the next begins. |
| `feature_list.json` | The names of all available features, used to check your selection |

`train` is needed to fit the model, `val` for early stopping and tuning, and `test` only for the final evaluation. Load `test` only when you are ready for that, to save memory.

### How to use it

In the cell marked **USER INPUT**:
1. Type the feature names you want into `FEATURES_TO_USE`, or set it to one of the ready-made lists in `PRESETS`. Names are checked against `feature_list.json`; a misspelt name stops the cell and suggests the closest correct names.
2. List the splits to load in `SPLITS_TO_LOAD`.
3. Optionally set `SESSION_FRACTION` below 1, to load a random subset of whole sessions (for quick experiments or if memory is short). Sessions are never split.
4. Set `AS_DATAFRAME = True` if you want pandas DataFrames (for inspection), or leave it `False` for NumPy arrays (for LightGBM; less memory).

Before loading, the cell prints the files it will read and the memory the result will need: rows × features × 4 bytes. All 90 features for the full training split need about 4.4 GB, and the 31 selected features about 1.5 GB.

### What it produces

A dictionary `DATA`, with one entry per split loaded:
- `X`: the features, as float32 (rows × selected features), or a DataFrame;
- `y`: labels 0–3;
- `group`: candidates per session;
- `session_id`: the session of each row.

It also produces `FEATS`, the list of selected feature names in column order. The rows keep the files' order, so `group` stays aligned with `X` and `y`.

In [22]:
# ============================================================================================
#                                       USER INPUT
# ============================================================================================
FEATURES_TO_USE = [
    # type feature names here, one per line, e.g.:
    "item_impr_train", "item_ctr_train", "brand_ctr_train",
    "i2v_cos_max", "i2v_cos_mean", "i2v_cos_last5", "last_i2v_cos",
    "title_cos_max", "title_cos_profile", "title_cos_profile_ord", "last_title_cos",
    "shop_aff", "shop_ord_cnt", "shop_last_seen_days", "brand_aff", "brand_ord_cnt",
    "exact_wid_last_seen_days", "qhist_wid_cnt", "cate3_slate_share",
]
# ...or use a ready-made list instead (uncomment one line):
# FEATURES_TO_USE = "all_90"
# FEATURES_TO_USE = "selected_31"
# FEATURES_TO_USE = "personal_only"
# FEATURES_TO_USE = ["family:B", "family:C", "family:D"]      # whole families can be mixed with names

SPLITS_TO_LOAD = ["train", "val"]       # add "test" only for the final evaluation
SESSION_FRACTION = 1.0                  # e.g. 0.25 to load a random quarter of the sessions
AS_DATAFRAME = False                    # True -> pandas DataFrames; False -> NumPy arrays (for LightGBM)
# ============================================================================================

In [ ]:
import difflib, json
from pathlib import Path
import numpy as np, pandas as pd
import pyarrow.parquet as pq

FEAT_DIR = Path("/content/drive/MyDrive/BT4222 Final/JDSearch/processed/lgbm_features_all")
ALL_FEATURES = json.loads((FEAT_DIR / "feature_list.json").read_text())
FAMILY_TABLE = pd.read_csv(FEAT_DIR / "feature_dictionary_part3.csv").set_index("feature")["family"]
POPULARITY = ["item_impr_train", "item_ctr_train", "brand_ctr_train", "cate3_ctr_train",
              "item_ctr_train_slate_pct", "item_ctr_train_slate_z"]
PRESETS = {
    "all_90": ALL_FEATURES,
    "selected_31": ["item_impr_train", "i2v_cos_max_slate_pct", "i2v_cos_max", "i2v_cos_max_slate_z", "i2v_cos_last5", "i2v_cos_mean", "title_cos_max", "title_cos_max_slate_z", "last_i2v_cos", "title_cos_max_slate_pct", "title_cos_profile", "title_cos_profile_slate_z", "title_cos_profile_slate_pct", "brand_ctr_train", "shop_share", "shop_aff", "last_title_cos", "shop_ord_cnt", "shop_aff_slate_pct", "shop_last_seen_days", "exact_wid_last_seen_days", "brand_aff", "brand_share", "title_cos_profile_ord", "brand_ord_cnt", "shop_aff_slate_z", "item_ctr_train_slate_z", "item_ctr_train", "item_ctr_train_slate_pct", "cate3_slate_share", "qhist_wid_cnt"],
    "personal_only": [f for f in ALL_FEATURES if f not in POPULARITY],   # without the population-level priors
}


def resolve_features(spec):
    # Turn the user's selection (names, presets, 'family:X') into a validated, de-duplicated list in file order.
    if isinstance(spec, str):
        spec = PRESETS[spec] if spec in PRESETS else [spec]
    chosen = []
    for item in spec:
        if item in PRESETS:
            chosen += PRESETS[item]
        elif item.startswith("family:"):
            fam = item.split(":", 1)[1]
            members = FAMILY_TABLE.index[FAMILY_TABLE == fam].tolist()
            if not members:
                raise ValueError(f"Unknown family '{fam}'. Families: {sorted(FAMILY_TABLE.unique())}")
            chosen += members
        elif item in ALL_FEATURES:
            chosen.append(item)
        else:
            raise ValueError(f"Unknown feature '{item}'. Did you mean: {difflib.get_close_matches(item, ALL_FEATURES, n=3)}?")
    chosen = set(chosen)
    return [f for f in ALL_FEATURES if f in chosen]        # keep the files' column order


def load_split(split, feats, session_fraction=1.0, seed=42, as_dataframe=False):
    # Read only the chosen columns of {split}.parquet into a preallocated float32 array (no doubling of memory).
    pf = pq.ParquetFile(FEAT_DIR / f"{split}.parquet")
    group = np.load(FEAT_DIR / f"{split}_groups.npy")
    keep_sess = np.ones(len(group), bool)
    if session_fraction < 1.0:                              # keep whole sessions only
        keep_sess = np.random.default_rng(seed).random(len(group)) < session_fraction
    keep_row = np.repeat(keep_sess, group)
    n = int(keep_row.sum())
    X = np.empty((n, len(feats)), dtype=np.float32)
    y = np.empty(n, dtype=np.int8)
    sid = np.empty(n, dtype=np.int32)
    pos_in, pos_out = 0, 0
    for batch in pf.iter_batches(batch_size=500_000, columns=["session_id", "label"] + feats):
        m = keep_row[pos_in:pos_in + batch.num_rows]
        k = int(m.sum())
        if k:
            cols = batch.to_pandas()
            X[pos_out:pos_out + k] = cols[feats].to_numpy(np.float32)[m]
            y[pos_out:pos_out + k] = cols["label"].to_numpy()[m]
            sid[pos_out:pos_out + k] = cols["session_id"].to_numpy()[m]
        pos_in += batch.num_rows
        pos_out += k
    assert pos_in == len(keep_row) and pos_out == n
    out = {"X": pd.DataFrame(X, columns=feats, copy=False) if as_dataframe else X,
           "y": y, "group": group[keep_sess], "session_id": sid}
    assert out["group"].sum() == n
    return out


FEATS = resolve_features(FEATURES_TO_USE)
print(f"{len(FEATS)} features selected:", FEATS)
print("\nFiles to be read:")
plan = []
for s in SPLITS_TO_LOAD:
    rows = pq.ParquetFile(FEAT_DIR / f"{s}.parquet").metadata.num_rows
    plan.append({"split": s, "data file": f"{s}.parquet", "groups file": f"{s}_groups.npy",
                 "rows (all sessions)": rows,
                 "approx. RAM (GB)": round(rows * SESSION_FRACTION * (len(FEATS) * 4 + 5) / 1e9, 2)})
display(pd.DataFrame(plan))

DATA = {}
for s in SPLITS_TO_LOAD:
    DATA[s] = load_split(s, FEATS, SESSION_FRACTION, as_dataframe=AS_DATAFRAME)
    print(f"{s}: X {DATA[s]['X'].shape}, sessions {len(DATA[s]['group']):,}")

19 features selected: ['exact_wid_last_seen_days', 'brand_aff', 'brand_ord_cnt', 'shop_aff', 'shop_last_seen_days', 'shop_ord_cnt', 'title_cos_profile', 'title_cos_profile_ord', 'title_cos_max', 'i2v_cos_mean', 'i2v_cos_max', 'i2v_cos_last5', 'last_title_cos', 'last_i2v_cos', 'qhist_wid_cnt', 'item_ctr_train', 'brand_ctr_train', 'item_impr_train', 'cate3_slate_share']

Files to be read:


,split,data file,groups file,rows (all sessions),approx. RAM (GB)
0,train,train.parquet,train_groups.npy,12279873,0.99
1,val,val.parquet,val_groups.npy,1536651,0.12


## Guide — training LightGBM on the loaded features

This guide continues from the loader. It is a guide only; no model is trained in this notebook.

### 1. What LightGBM needs for ranking

LightGBM's `lambdarank` objective learns to order the candidates within each session. For each split, it needs:
- **X**, the feature matrix: `DATA[split]["X"]`;
- **y**, the labels 0 to 3: `DATA[split]["y"]`. Here 0 = ignored, 1 = click, 2 = cart, 3 = order;
- **group**, the candidates per session, in row order: `DATA[split]["group"]`.

Do not shuffle or re-sort the rows after loading. `group` describes them in the order they were loaded.

### 2. Building the Datasets

```python
import lightgbm as lgb

dtrain = lgb.Dataset(DATA["train"]["X"], DATA["train"]["y"], group=DATA["train"]["group"],
                     feature_name=FEATS, free_raw_data=True)
dval = lgb.Dataset(DATA["val"]["X"], DATA["val"]["y"], group=DATA["val"]["group"], reference=dtrain)
```

`reference=dtrain` makes the validation data use the same bins as the training data. LightGBM bins each feature into at most 255 values internally, so its own copy is about a quarter of the size of the float32 array. After `dtrain.construct()`, the loaded array can be deleted (`del DATA["train"]["X"]`) to free memory.

To avoid loading and binning again in later sessions, save the Dataset once in LightGBM's binary format. It stores the bins, labels and groups, and loads quickly:

```python
dtrain.save_binary(str(FEAT_DIR / "train_selected.bin"))      # once
dtrain = lgb.Dataset(str(FEAT_DIR / "train_selected.bin"))     # later sessions
```

A binary file is tied to the feature selection it was built with. Build a new one when you change `FEATURES_TO_USE`.

### 3. Training

```python
params = dict(
    objective="lambdarank", metric="ndcg", eval_at=[10],
    label_gain=[0, 1, 3, 7],          # gains for labels 0-3: an order counts more than a click
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    seed=42, verbose=-1,
)
ranker = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dval], valid_names=["val"],
                   callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)])
ranker.save_model(str(FEAT_DIR / "lgbm_ranker.txt"))
```

- Validation NDCG@10 is reported after every round, and early stopping keeps the best number of rounds.
- Missing values (`NaN`) are left as they are. LightGBM sends them to whichever side of each split works best, and here "missing" means "no evidence".
- Tune `learning_rate`, `num_leaves`, `min_data_in_leaf` and the subsampling fractions on the validation split only.
- With all 90 features, the full training matrix needs about 4.4 GB of RAM before binning. If that is too much, set `SESSION_FRACTION` to 0.5 for training, or use a smaller feature selection.

### 4. Evaluating on the test split (once, at the end)

Load `test` with the same `FEATURES_TO_USE` (add `"test"` to `SPLITS_TO_LOAD` and rerun the loader). Then compute the metrics as step_2_query_popularity_baseline_models.ipynb's `evaluate_ranking` does: binary relevance (label > 0), ties broken at random with seed 42, NDCG@10, MRR and P@1.

```python
def ranking_metrics(session_id, score, label, k=10, seed=42):
    np.random.seed(seed); tb = np.random.rand(len(score))
    s = np.where(np.isnan(score), -np.inf, score)
    order = np.lexsort((-tb, -s, session_id))
    sid, rel = session_id[order], label[order] > 0
    first = np.r_[True, sid[1:] != sid[:-1]]; grp = np.cumsum(first) - 1
    rank = np.arange(len(sid)) - np.flatnonzero(first)[grp] + 1
    dcg = np.bincount(grp, weights=np.where(rel & (rank <= k), 1 / np.log2(rank + 1), 0))
    n_rel = np.bincount(grp, weights=rel)
    idcg = np.r_[0, np.cumsum(1 / np.log2(np.arange(2, k + 2)))][np.minimum(n_rel, k).astype(int)]
    best = np.full(grp[-1] + 1, np.inf); np.minimum.at(best, grp[rel], rank[rel])
    has = n_rel > 0
    return {"NDCG@10": (dcg[has] / idcg[has]).mean(), "MRR": (1 / best[has]).mean(), "P@1": (best[has] == 1).mean()}

te = DATA["test"]
print("LightGBM:", ranking_metrics(te["session_id"], ranker.predict(te["X"], num_iteration=ranker.best_iteration), te["y"]))
if "item_ctr_train" in FEATS:
    print("Popularity baseline:", ranking_metrics(te["session_id"], te["X"][:, FEATS.index("item_ctr_train")], te["y"]))
```

The popularity baseline (`item_ctr_train`, computed from training sessions only) is the leakage-free reference that the model should beat.

### 5. Comparing feature sets

Because the loader selects features by name, comparing feature sets only means changing `FEATURES_TO_USE` and retraining:
- **`"all_90"` against `"selected_31"`:** does the full set add anything over the 31 strongest features?
- **`"all_90"` against `"personal_only"`:** how much of the model's performance comes from personal similarity rather than population popularity?
- **One family at a time:** for example, all 90 minus `"family:D"`, to measure the value of the item2vec features. Compare validation NDCG@10 between runs, and use the test split only for the final choice.

### 6. Interpreting the model

- **Feature importance.** `pd.Series(ranker.feature_importance("gain"), index=FEATS).sort_values()` shows which features the trees rely on. Several features are near-duplicates (for example `shop_aff` and `shop_share`, or a `_slate_z` feature and its base), so importance is shared between them. Group importances by family (`FAMILY_TABLE`) before drawing conclusions.
- **Where the gains come from.** Break test NDCG@10 down by session type: sessions with an exact match (`exact_wid_last_seen_days` present for some candidate), sessions with only a shop or brand match, and sessions with neither. This shows whether the similarity features help where they have something to measure.